# 03 — Train POSTER++ stretch model

**Reproduce-first protocol:** load the released RAF-DB checkpoint and verify ≥ 91% WAR on the test split BEFORE attempting any fine-tuning. If reproduction fails, debug environment (PyTorch version, IR-50/MobileFaceNet backbone weights). report.md:933–934 — never train from random init.

In [ ]:
# === Bootstrap: mount Drive, clone repo, hydrate data, link runs/ to Drive ===
# Requires Colab "Secrets" entry GH_TOKEN with a fine-grained PAT for radudeaconu/fer.
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not Path('/content/fer').exists():
    os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
    !git clone https://$GH_TOKEN@github.com/radudeaconu/fer.git /content/fer
%cd /content/fer
!pip install -q -r requirements.txt
%run scripts/colab_bootstrap.py

# POSTER++-specific: clone the upstream repo for backbone definitions.
import os
os.makedirs('third_party', exist_ok=True)
if not os.path.exists('third_party/POSTER_V2'):
    !git clone https://github.com/Talented-Q/POSTER_V2.git third_party/POSTER_V2


In [ ]:
# POSTER++ requires IR-50 + MobileFaceNet backbones AND the RAF-DB ckpt.
# Place all three under MyDrive/fer-data/checkpoints/ and copy locally.
import shutil
from pathlib import Path
Path('checkpoints').mkdir(exist_ok=True)
Path('third_party/POSTER_V2/models/pretrain').mkdir(parents=True, exist_ok=True)

DRIVE = Path('/content/drive/MyDrive/fer-data/checkpoints')
for src_name, dst in [
    ('poster_v2_rafdb.pth', Path('checkpoints/poster_v2_rafdb.pth')),
    ('ir50.pth', Path('third_party/POSTER_V2/models/pretrain/ir50.pth')),
    ('mobilefacenet_model_best.pth.tar', Path('third_party/POSTER_V2/models/pretrain/mobilefacenet_model_best.pth.tar')),
]:
    src = DRIVE / src_name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst); print(f'Copied {src_name} -> {dst}')
    elif dst.exists():
        print(f'{dst} already present')
    else:
        print(f'WARNING: missing {src}')

## Reproduce released checkpoint (no training)
Expect WAR ≥ 91%. If lower, stop and debug — do not proceed to fine-tuning.

In [ ]:
!python -m src.eval --config configs/poster_rafdb.yaml --ckpt checkpoints/poster_v2_rafdb.pth

## Fine-tune (only proceed if reproduction succeeded)

In [ ]:
!python -m src.train --config configs/poster_rafdb.yaml

In [ ]:
!python -m src.eval --config configs/poster_rafdb.yaml --ckpt runs/poster_rafdb/best.pth